# CNN over wav2vec2 frames — Google Colab

**Run this on Colab, not locally.** It needs a working PyTorch, and Windows
Smart App Control blocks torch on the machine this project was developed on
(`docs/environment.md`). Colab also gives a free GPU, which turns a ~40 minute
feature extraction into about two.

### What this does

The project's best model pools wav2vec2's output over time — it averages every
frame of a clip into one vector, then fits a linear classifier. That throws away
*when* things happen. This notebook keeps the time axis and runs a 1-D
convolutional network over it, so the model can learn that a rise in pitch
followed by a drop means something different from the reverse.

### What it reuses, and why that matters

It imports the project's own `ser` package for three things:

- **`splits/folds.csv`** — the frozen train/test split. Using the same actor
  assignment is the only reason a number from here is comparable to one from
  the main project.
- **`ser.evaluate`** — the same metrics, computed the same way.
- **`ser.preprocess.load_audio`** — the single audio entry point, so the clips
  are decoded at 16 kHz mono exactly as they were elsewhere.

### First: turn on the GPU

**Runtime → Change runtime type → Hardware accelerator → T4 GPU**, then run the
cell below. Everything works on CPU too, just several times slower.

In [ ]:
import torch, subprocess
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("\nNo GPU. Runtime > Change runtime type > T4 GPU, then rerun.")
    print("It will still work on CPU, roughly 10x slower.")

## 1. Install what Colab is missing

Colab ships torch, numpy, pandas and sklearn. It needs `transformers` for
wav2vec2 and `librosa`/`soundfile` for audio decoding.

In [ ]:
%pip install -q transformers librosa soundfile
print("installed")

## 2. Upload the project

You need two files from the handover:

| File | What |
|---|---|
| `code.zip` | the `ser` package, `splits/folds.csv`, `results/results.csv` |
| `Unheard-data.zip` | the audio (only RAVDESS is needed here) |

Run the cell and pick both. For a ~1 GB data file the browser upload is slow —
if it times out, put the zips in Google Drive and use the Drive cell instead.

In [ ]:
from google.colab import files
uploaded = files.upload()          # select code.zip and Unheard-data.zip
print("\nuploaded:", list(uploaded))

**Alternative — from Google Drive** (faster and survives a disconnect):

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/code.zip" /content/
# !cp "/content/drive/MyDrive/Unheard-data.zip" /content/

## 3. Unpack and check

In [ ]:
import os, sys, zipfile, pathlib

ROOT = pathlib.Path("/content/Unheard")
ROOT.mkdir(exist_ok=True)

for z, dest in [("code.zip", ROOT), ("Unheard-data.zip", ROOT)]:
    if pathlib.Path(z).exists():
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
        print(f"unpacked {z}")

# code.zip may unpack into a nested folder; find the one holding ser/
if not (ROOT / "ser").is_dir():
    for child in ROOT.rglob("ser/metadata.py"):
        ROOT = child.parent.parent
        break

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("project root:", ROOT)
print("ser/ present    :", (ROOT / "ser").is_dir())
print("folds.csv       :", (ROOT / "splits" / "folds.csv").is_file())
print("audio present   :", (ROOT / "data" / "data" / "Person01").is_dir())

In [ ]:
from ser import SEED, set_seeds
from ser.metadata import load as load_metadata
from ser.splits import get_split, load_folds

set_seeds()
meta = load_metadata()
folds = load_folds()
print(f"{len(meta)} clips, {meta.actor.nunique()} actors")
print(meta.emotion_label.value_counts().to_string())

## 4. Extract wav2vec2 frame-level features

This is the part that differs from the main project. `ser.embeddings.extract`
returns the **mean and standard deviation over the whole clip** — one vector per
clip, no time axis. A convolution has nothing to slide along in that.

`extract_frames` keeps every frame: roughly 180 frames of 768 numbers for a 3.7
second clip, at 50 frames per second.

Clips vary from 2.94 to 5.27 seconds, so they are padded to a common length and
the true length of each is recorded. **The padding must be masked** or the
network will learn from silence that was never in the recording.

In [ ]:
import numpy as np, time, torch
from transformers import AutoModel, AutoFeatureExtractor
from ser.preprocess import load_audio, TARGET_SR

MODEL_ID = "facebook/wav2vec2-base"
LAYER    = 6          # the project's best layer; see docs/wav2vec2_findings.md
MAX_FRAMES = 280      # ~5.6 s at 50 fps, above RAVDESS's longest clip
BATCH    = 16

device = "cuda" if torch.cuda.is_available() else "cpu"
fe    = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
assert fe.sampling_rate == TARGET_SR

paths = meta.filepath.tolist()
X = np.zeros((len(paths), MAX_FRAMES, 768), dtype=np.float32)
L = np.zeros(len(paths), dtype=np.int32)

t0 = time.time()
for s in range(0, len(paths), BATCH):
    chunk = paths[s:s + BATCH]
    sigs  = [load_audio(p)[0] for p in chunk]
    inp   = fe(sigs, sampling_rate=TARGET_SR, return_tensors="pt",
               padding=True, return_attention_mask=True).to(device)
    with torch.inference_mode():
        hs = model(**inp, output_hidden_states=True).hidden_states[LAYER]
    lens = model._get_feat_extract_output_lengths(
        inp["attention_mask"].sum(-1)).cpu().numpy()
    hs = hs.float().cpu().numpy()
    for i in range(len(chunk)):
        n = min(int(lens[i]), MAX_FRAMES)
        X[s + i, :n] = hs[i, :n]
        L[s + i] = n
    if s % (BATCH * 20) == 0:
        print(f"  {s + len(chunk):>5}/{len(paths)}", flush=True)

print(f"\ndone in {(time.time()-t0)/60:.1f} min")
print("X:", X.shape, "| frames per clip:", L.min(), "-", L.max())
np.save("/content/frames.npy", X); np.save("/content/lengths.npy", L)
print("cached to /content — survives a kernel restart, not a disconnect")

## 5. The network

A 1-D CNN over time. Each of the 768 wav2vec2 dimensions is an input channel;
convolutions slide along frames, so a filter sees a short window of time — about
100 ms at kernel size 5.

Three points worth understanding rather than copying:

**Masked pooling.** After the convolutions, the frames are averaged into one
vector — but only the real ones. `mask` zeroes the padding before summing, and
the divisor is the true length. Without this, a short clip's average is diluted
by however much silence it was padded with.

**Small on purpose.** There are 1,140 training clips. Anything large memorises
them. Two conv blocks and a 128-unit head is already generous; in the main
project a *linear* classifier beat XGBoost by 13 points, which says this dataset
does not support much capacity.

**Per-speaker normalisation.** Worth +16 pp in the main project — every speaker's
features are expressed relative to their own average. It is applied here too,
across the frame axis.

In [ ]:
import torch.nn as nn

class EmotionCNN(nn.Module):
    def __init__(self, in_dim=768, n_classes=8, width=128, p_drop=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_dim, width, kernel_size=5, padding=2),
            nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(p_drop),
            nn.Conv1d(width, width, kernel_size=5, padding=2),
            nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(p_drop),
        )
        self.head = nn.Linear(width, n_classes)

    def forward(self, x, lengths):
        # x: (batch, frames, 768) -> conv1d wants (batch, channels, frames)
        h = self.conv(x.transpose(1, 2))                    # (B, width, T)
        mask = (torch.arange(x.shape[1], device=x.device)[None, :]
                < lengths[:, None]).float().unsqueeze(1)    # (B, 1, T)
        h = (h * mask).sum(-1) / mask.sum(-1).clamp(min=1)  # masked mean
        return self.head(h)

print(EmotionCNN())
print("\nparameters:", sum(p.numel() for p in EmotionCNN().parameters()))

## 6. Train across the frozen folds

Five folds, each training on the actors the project assigns to training and
testing on actors the model has never heard. Predictions from all five are
pooled and scored once — the same procedure as everywhere else in the project,
so the number lands on the same scale.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder

EPOCHS, LR, BATCH_TRAIN = 40, 1e-3, 32

pos = {f: i for i, f in enumerate(meta.filename)}
speakers = meta.actor.to_numpy()
enc = LabelEncoder().fit(meta.emotion_label)

# per-speaker normalisation, across the frame axis
Xn = X.copy()
for a in np.unique(speakers):
    rows = np.flatnonzero(speakers == a)
    flat = np.concatenate([Xn[i, :L[i]] for i in rows])
    mu, sd = flat.mean(0), flat.std(0) + 1e-8
    for i in rows:
        Xn[i, :L[i]] = (Xn[i, :L[i]] - mu) / sd

def to_tensors(names):
    idx = [pos[n] for n in names]
    return (torch.tensor(Xn[idx]), torch.tensor(L[idx], dtype=torch.long),
            torch.tensor(enc.transform(meta.emotion_label.to_numpy()[idx])))

per_fold = []
for fold in range(5):
    tr, te = get_split("speaker_independent", fold, folds=folds)
    xtr, ltr, ytr = to_tensors(tr)
    xte, lte, yte = to_tensors(te)

    torch.manual_seed(SEED)
    net = EmotionCNN().to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-2)
    lossf = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(xtr, ltr, ytr),
                        batch_size=BATCH_TRAIN, shuffle=True)

    net.train()
    for ep in range(EPOCHS):
        tot = 0.0
        for xb, lb, yb in loader:
            xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
            opt.zero_grad()
            loss = lossf(net(xb, lb), yb)
            loss.backward(); opt.step()
            tot += loss.item() * len(xb)
        if (ep + 1) % 10 == 0:
            print(f"  fold {fold} epoch {ep+1:>2}  loss {tot/len(xtr):.4f}")

    net.eval()
    with torch.inference_mode():
        pred = net(xte.to(device), lte.to(device)).argmax(1).cpu().numpy()
    pred = enc.inverse_transform(pred)

    truth = meta.set_index("filename").loc[te, "emotion_label"].to_numpy()
    test_meta = meta.set_index("filename").loc[te].reset_index()
    per_fold.append((truth, pred, test_meta))
    print(f"  fold {fold}: accuracy {(truth == pred).mean():.4f}\n")

## 7. Score it the same way as everything else

In [ ]:
from ser.evaluate import evaluate_pooled

result = evaluate_pooled(
    per_fold, "speaker_independent",
    model="cnn1d-wav2vec2-L6", feature_config="wav2vec2-L6-frames-per_speaker",
    hyperparams={"epochs": EPOCHS, "lr": LR, "width": 128, "layer": LAYER},
    notes="colab: 1D CNN over wav2vec2 frame-level features",
)

print(f"accuracy  {result['accuracy']:.4f}")
print(f"macro-F1  {result['macro_f1']:.4f}")
print()
print("for comparison, from the main project:")
print("  0.8220  wav2vec2 L6 pooled + linear head (nested CV)")
print("  0.6311  handcrafted MFCC features + SVM")
print()
print(result["per_class"].round(3).to_string())

## 8. Is the difference real?

With 24 test speakers, differences under about 2 points are not distinguishable
from noise. Clips from one actor succeed or fail together, so the bootstrap
resamples **speakers**, not clips — treating 60 clips from one person as 60
independent observations would make the interval far too narrow.

In [ ]:
from ser.stats import cluster_bootstrap_ci
import pandas as pd

allp = pd.concat([pd.DataFrame({"y_true": t, "y_pred": p,
                                "speaker": m.actor.astype(str).to_numpy()})
                  for t, p, m in per_fold], ignore_index=True)

ci = cluster_bootstrap_ci(allp.y_true, allp.y_pred, allp.speaker, "macro_f1")
print(f"macro-F1 {ci['point']:.4f}   95% CI [{ci['lo']:.4f}, {ci['hi']:.4f}]")
print(f"\nthe pooled linear model is 0.8220 — inside this interval means the")
print(f"CNN has not been shown to differ from it, in either direction.")

## 9. Save the results

In [ ]:
from google.colab import files
allp.to_csv("/content/cnn_predictions.csv", index=False)
files.download("/content/cnn_predictions.csv")
files.download("/content/results/results.csv")   # the shared log, with the new rows

## If you want to push it further

**More epochs or a wider network will overfit before they help.** 1,140 training
clips is the binding constraint. Try these instead, roughly in order of expected
value:

1. **Augmentation** — the standard fix for too little data, and untried in this
   project. On frames: mask random time spans and random feature channels
   (SpecAugment). Training folds only.
2. **Attention pooling** instead of the masked mean — let the network learn which
   frames matter rather than weighting them equally. This is the natural next
   step from here and the reason keeping the time axis is worth anything.
3. **A different layer.** Layer 6 is best for the *pooled* model; the best layer
   for a CNN is an open question and the loop above changes with one constant.
4. **All four corpora.** `ser.arms` gives the exact four training setups compared
   in the main project — pretrain on all 12,162 clips then fine-tune on RAVDESS,
   rather than pooling them. See `docs/two_stage_findings.md`.

**Log every run** through `evaluate_pooled` so results stay in one place and
stay comparable.